# EDA

In [ ]:
data_dir = "dataMed_3/Drug Vision/Data Combined"


# EFFICIENCTNet + FINETuning

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the image in BGR format using OpenCV
img_md_bgr = cv2.imread("dataMed_3/Drug Vision/Data Combined/DayZinc/00000000.jpg", cv2.IMREAD_COLOR)

# Split the image into BGR channels
b, g, r = cv2.split(img_md_bgr)

# Create a larger figure for better visualization
plt.figure(figsize=(20, 6))

# Display Red Channel
plt.subplot(1, 4, 1)
plt.imshow(r, cmap='gray')
plt.title("Red Channel")
plt.axis('off')  # Hide axes for clarity

# Display Green Channel
plt.subplot(1, 4, 2)
plt.imshow(g, cmap='gray')
plt.title("Green Channel")
plt.axis('off')  # Hide axes for clarity

# Display Blue Channel
plt.subplot(1, 4, 3)
plt.imshow(b, cmap='gray')
plt.title("Blue Channel")
plt.axis('off')  # Hide axes for clarity

# Merge the channels back together
img_merged = cv2.merge((b, g, r))

# Convert the image back to RGB for correct color representation (OpenCV uses BGR)
img_merged_rgb = cv2.cvtColor(img_merged, cv2.COLOR_BGR2RGB)

# Display the merged image
plt.subplot(1, 4, 4)
plt.imshow(img_merged_rgb)
plt.title("Merged Image")
plt.axis('off')  # Hide axes for clarity

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import torch
import importlib
import cv2 
import pandas as pd

from PIL import Image
from IPython.display import display

TEST_IMAGE_PATH = "dataMed_/ImageClassesCombinedWithCOCOAnnotations/test_image.JPG"
img = cv2.imread(TEST_IMAGE_PATH)

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
display(Image.fromarray(img))

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
train_data= ImageDataGenerator(
    rescale=1. / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)
train_batch = train_data.flow_from_directory(
    directory=data_dir,
    target_size=(351,351),
    batch_size=16,
)

g_dict = train_batch.class_indices      # defines dictionary {'class': index}
classes = list(g_dict.keys())       # defines list of dictionary's kays (classes), classes names : string
images, labels = next(train_batch)      # get a batch size samples from the generator

# calculate number of displayed samples
length = len(labels)        # length of batch size
sample = min(length, 30)    # check if sample less than 25 images

plt.figure(figsize= (20, 20))

for i in range(sample):
    plt.subplot(5, 5, i + 1)
    image = images[i]      # scales data to range (0 - 255)
    plt.imshow(image)
    index = np.argmax(labels[i])  # get image index
    class_name = classes[index]   # get class of image
    plt.title(class_name, color= 'Blue', fontsize= 12)
    plt.axis('off')
plt.show()

In [ ]:
def plot_images_from_generator(generator, title, num_images=10, images_per_row=5):
    images, labels = next(generator)
    images = images[:num_images]
    labels = labels[:num_images]
    num_rows = (num_images + images_per_row - 1) // images_per_row
    fig, axes = plt.subplots(num_rows, images_per_row, figsize=(15, 3 * num_rows))
    fig.suptitle(title, fontsize=16)
    axes = axes.flatten()
    for i in range(num_images):
        img = images[i]
        label = labels[i]
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f"Label: {list(generator.class_indices.keys())[label.argmax()]}")
    
    for j in range(num_images, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

plot_images_from_generator(train_batch, "train_data", num_images=10, images_per_row=5)



In [ ]:
import tensorflow as tf

In [ ]:
# data_dir = "dataMed_3/Drug Vision/Data Combined"

IMAGE_SHAPE = (224,224)
BATCH_SIZE= 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(data_dir , image_size = IMAGE_SHAPE , batch_size = BATCH_SIZE , label_mode = "categorical" , 
                                                                validation_split = 0.2 , subset = "training" , seed = 42)

test_data = tf.keras.preprocessing.image_dataset_from_directory(data_dir , image_size = IMAGE_SHAPE , batch_size = BATCH_SIZE , label_mode = "categorical" , 
                                                                validation_split = 0.2 , subset = "validation" , seed = 42)


In [ ]:
train_data.class_names

In [ ]:
class_distribution = [len(os.listdir(os.path.join(data_dir, name))) for name in class_names]
print(class_distribution)


In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import os

In [ ]:

# Define a transform to normalize the images (if needed)
transform = transforms.Compose([transforms.ToTensor()])

# Load the dataset using ImageFolder
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Get the class names from the dataset
class_names = dataset.classes

# Count the number of samples per class
class_counts = [0] * len(class_names)

for _, label in dataset:
    class_counts[label] += 1

# Plot the bar graph
plt.figure(figsize=(15,6))
plt.bar(class_names, class_counts, color='green')
plt.xlabel('Classes')
plt.ylabel('Number of Samples')
plt.title('Number of Samples per Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# BASE MODEL - EfficientNet

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top = False)

# DATA AUGMENTATION LAYER

In [ ]:
from tensorflow.keras import layers

data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomHeight(0.2),
    layers.RandomWidth(0.2),
])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random

In [ ]:
#random image with augmentation

target_class = random.choice(train_data.class_names)
target_dir = "dataMed_3/Drug Vision/Data Combined/" + target_class
random_image = random.choice(os.listdir(target_dir))
random_image_path = target_dir + "/" + random_image


img = mpimg.imread(random_image_path)
plt.imshow(img)
plt.title(f"Original random image from class: {target_class}")
plt.axis(False);



augmented_img = data_aug(tf.expand_dims(img, axis=0))
plt.figure()
plt.imshow(tf.squeeze(augmented_img)/255.)
plt.title(f"Augmented random image from class: {target_class}")
plt.axis(False);

In [ ]:
input_shape = (224,224,3)

base_model.trainable = False


inputs = tf.keras.layers.Input(shape = input_shape)

x = data_aug(inputs)

x = base_model(x , training = False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

outputs = tf.keras.layers.Dense(10 , activation = "softmax")(x)

model = tf.keras.Model(inputs , outputs)

model.compile(loss = "categorical_crossentropy",
              optimizer = tf.keras.optimizers.Adam(),
              metrics = ["accuracy"])

history = model.fit(train_data , 
                    epochs = 5 ,
                    steps_per_epoch = len(train_data),
                    validation_data = test_data,
                    validation_steps = int(0.25 * len(test_data))
                    )

In [ ]:
import seaborn as sns
sns.set_style("darkgrid")

def plot_model(history):
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]
    
    epochs = len(loss)
    
    fig,axes = plt.subplots(nrows = 2, ncols=1 , figsize = (14,14))
    axes[0].plot(accuracy , label = "training accuracy")
    axes[0].plot(val_accuracy , label = "validation accuracy")
    
    axes[1].plot(loss , label = "training loss")
    axes[1].plot(val_loss , label = "validation loss")
    
    axes[0].legend()
    axes[1].legend()
    
    plt.xlabel("epochs")
    
    axes[0].set_title("Accuracy")
    axes[1].set_title("Loss")
    
    plt.show()

In [ ]:
plot_model(history)

      FINE TUNING - CONTINUE TRAINING MODEL FOR ANOTHER 5 EPOCHS

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-10]:
    layer.trainable = False
    
model.compile(loss = "categorical_crossentropy",
              optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001),
              metrics = ["accuracy"])

In [ ]:
len(model.trainable_variables)

In [ ]:
initial_epoch = 5
fine_tuning_epoch = initial_epoch + 5

history_finetuned = model.fit(train_data,
                              epochs = fine_tuning_epoch,
                              validation_data = test_data,
                              validation_steps = int(0.25 * len(test_data)),
                              initial_epoch = history.epoch[-1])

In [ ]:
model.evaluate(test_data)